In [27]:
import torch
import torch.nn as nn
import torchvision
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mediapipe as mp
import math
import cv2

from torch.utils.data import TensorDataset, DataLoader, Dataset
from torchvision import transforms
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
from typing import Tuple, Union
from PIL import Image
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


In [19]:
all_models = models.list_models(module=torchvision.models)
print(all_models)

all_weights = models.get_model_weights("vgg19")
print(*all_weights)

['alexnet', 'convnext_base', 'convnext_large', 'convnext_small', 'convnext_tiny', 'densenet121', 'densenet161', 'densenet169', 'densenet201', 'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3', 'efficientnet_b4', 'efficientnet_b5', 'efficientnet_b6', 'efficientnet_b7', 'efficientnet_v2_l', 'efficientnet_v2_m', 'efficientnet_v2_s', 'googlenet', 'inception_v3', 'maxvit_t', 'mnasnet0_5', 'mnasnet0_75', 'mnasnet1_0', 'mnasnet1_3', 'mobilenet_v2', 'mobilenet_v3_large', 'mobilenet_v3_small', 'regnet_x_16gf', 'regnet_x_1_6gf', 'regnet_x_32gf', 'regnet_x_3_2gf', 'regnet_x_400mf', 'regnet_x_800mf', 'regnet_x_8gf', 'regnet_y_128gf', 'regnet_y_16gf', 'regnet_y_1_6gf', 'regnet_y_32gf', 'regnet_y_3_2gf', 'regnet_y_400mf', 'regnet_y_800mf', 'regnet_y_8gf', 'resnet101', 'resnet152', 'resnet18', 'resnet34', 'resnet50', 'resnext101_32x8d', 'resnext101_64x4d', 'resnext50_32x4d', 'shufflenet_v2_x0_5', 'shufflenet_v2_x1_0', 'shufflenet_v2_x1_5', 'shufflenet_v2_x2_0', 'squeezenet1_0

In [44]:
#!unzip /content/KinshipVerification.zip

### Config

In [20]:
parent_dir = Path("KinshipVerification/Parents")
child_dir = Path("KinshipVerification/Children")
face_detection_model_path = "blaze_face_short_range.tflite"
face_detection_padding = 20  # pixels
image_resize_dim = (224, 224)  # (width, height)
overwrite_existing = False  # Set to True to overwrite existing preprocessed images

MARGIN = 10  # pixels
ROW_SIZE = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
TEXT_COLOR = (255, 0, 0)  # red

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Face Detection Visualization Functions

In [6]:
def _normalized_to_pixel_coordinates(
    normalized_x: float, normalized_y: float, image_width: int,
    image_height: int) -> Union[None, Tuple[int, int]]:
  """Converts normalized value pair to pixel coordinates."""

  # Checks if the float value is between 0 and 1.
  def is_valid_normalized_value(value: float) -> bool:
    return (value > 0 or math.isclose(0, value)) and (value < 1 or
                                                      math.isclose(1, value))

  if not (is_valid_normalized_value(normalized_x) and
          is_valid_normalized_value(normalized_y)):
    # TODO: Draw coordinates even if it's outside of the image bounds.
    return None
  x_px = min(math.floor(normalized_x * image_width), image_width - 1)
  y_px = min(math.floor(normalized_y * image_height), image_height - 1)
  return x_px, y_px


def visualize(
    image,
    detection_result
) -> np.ndarray:
  """Draws bounding boxes and keypoints on the input image and return it.
  Args:
    image: The input RGB image.
    detection_result: The list of all "Detection" entities to be visualize.
  Returns:
    Image with bounding boxes.
  """
  annotated_image = image.copy()
  height, width, _ = image.shape

  for detection in detection_result.detections:
    # Draw bounding_box
    bbox = detection.bounding_box
    start_point = bbox.origin_x, bbox.origin_y
    end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
    cv2.rectangle(annotated_image, start_point, end_point, TEXT_COLOR, 3)

    # Draw keypoints
    for keypoint in detection.keypoints:
      keypoint_px = _normalized_to_pixel_coordinates(keypoint.x, keypoint.y,
                                                     width, height)
      color, thickness, radius = (0, 255, 0), 2, 2
      cv2.circle(annotated_image, keypoint_px, thickness, color, radius)

    # Draw label and score
    category = detection.categories[0]
    category_name = category.category_name
    category_name = '' if category_name is None else category_name
    probability = round(category.score, 2)
    result_text = category_name + ' (' + str(probability) + ')'
    text_location = (MARGIN + bbox.origin_x,
                     MARGIN + ROW_SIZE + bbox.origin_y)
    cv2.putText(annotated_image, result_text, text_location, cv2.FONT_HERSHEY_PLAIN,
                FONT_SIZE, TEXT_COLOR, FONT_THICKNESS)

  return annotated_image

### Preprocessing Pipeline
- Currently only does cropping using BlazeFace (short-range) for facial detection

In [ ]:
class preprocess_pipeline():
    def __init__(self, parent_dir, child_dir, face_detection_model_path, padding=20, size=(224, 224), visualize=False, overwrite_existing=False):
        if(overwrite_existing):
            print("Warning: Existing preprocessed images will be overwritten.")
        
            parents = sorted(parent_dir.glob("*")) # Sort the file paths to ensure consistent ordering
            children = sorted(child_dir.glob("*"))
            preprocessed_images_parents = self.preprocess_image(parents, face_detection_model_path, padding=padding, size=size, visualize=visualize)
            preprocessed_images_children = self.preprocess_image(children, face_detection_model_path, padding=padding, size=size, visualize=visualize)
            self.save_to_folder(parents, preprocessed_images_parents, "Preprocessed/Parents", overwrite=overwrite_existing)
            self.save_to_folder(children, preprocessed_images_children, "Preprocessed/Children", overwrite=overwrite_existing)
        print("Preprocessing already completed. Set overwrite_existing=True to re-run preprocessing and overwrite existing images.")

    def is_valid_image(self, image_path):
        try:
            with Image.open(image_path) as img:
                img.verify()  # Verify that it's an image
            return True
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            return False
    
    def can_load_image(self, image_path):
        try:
            with Image.open(image_path) as img:
                img.load()  # Attempt to load the image data
            return True
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return False
    
    def detect_face(self, image_path, face_detection_model_path,visualize=False):
        if not self.is_valid_image(image_path):
            print(f"Skipping invalid image: {image_path}")
            return None
        
        if not self.can_load_image(image_path):
            print(f"Skipping image that cannot be loaded: {image_path}")
            return None

        base_options = python.BaseOptions(model_asset_path=face_detection_model_path)
        options = vision.FaceDetectorOptions(base_options=base_options,
                                             running_mode=vision.RunningMode.IMAGE,
                                             min_detection_confidence=0.5)
        detector = vision.FaceDetector.create_from_options(options)

        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        result = detector.detect(mp_image)

        if len(result.detections) == 0:
            print(f"No face detected in image: {image_path}")
            return None
        
        if visualize:
             self.visualize_detection(mp_image, result)

        return result
            
    
    def visualize_detection(self, mp_image, detection_result):
        image_copy = np.copy(mp_image.numpy_view())
        annotated_image = visualize(image_copy, detection_result)

        plt.imshow(annotated_image)
        plt.axis("off")
        plt.show()

    def crop_face(self, image_path, detection_result, padding=20):
        image = cv2.imread(image_path)
        bbox = detection_result.detections[0].bounding_box # Assuming we take the first detected face (only one face per image)
        
        x, y = bbox.origin_x, bbox.origin_y
        box_width, box_height = bbox.width, bbox.height


        x1 = max(0, x - padding + 10) # Ensure we don't go below 0
        y1 = max(0, y - padding)
        x2 = min(x + box_width + padding - 10, image.shape[1]) # Ensure we don't go beyond image width
        y2 = min(y + box_height + padding, image.shape[0]) # Ensure we don't go beyond image height
        cropped_face = image[y1:y2, x1:x2]
        return cropped_face
    
    def align_face(self, image, detection_result):
        # Placeholder for face alignment logic
        pass
    
    # Will replace with super-resolution in future iterations, will use torchvision to resize for now
    # def resize_image(self, image, size=(224, 224)):
    #     # 1. Upsample
    #     upscaled = cv2.resize(image, (128, 128), interpolation=cv2.INTER_LANCZOS4)

    #     # 2. Blur the upscaled image
    #     blur = cv2.GaussianBlur(upscaled, (0, 0), sigmaX=1.0)

    #     # 3. Sharpen
    #     sharpened = cv2.addWeighted(upscaled, 1.5, blur, -0.5, 0)
    #     return sharpened
    
    def preprocess_image(self, image_list, face_detection_model_path, padding=20, size=(224, 224), visualize=False):
        processed_images = []
        for image_path in image_list:
            detection_result = self.detect_face(image_path, face_detection_model_path, visualize=visualize)
            if detection_result is None:
                continue

            cropped_face = self.crop_face(image_path, detection_result, padding=padding)
            #resized_face = self.resize_image(cropped_face, size=size)
            processed_images.append(cropped_face)

        return processed_images
    
    def save_to_folder(self, image_paths, processed_images, output_folder, overwrite=False):
        if(Path(output_folder).exists() and not overwrite):
            print(f"Output folder '{output_folder}' already exists. Skipping save to avoid overwriting.")
            return

        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)

        for path, image in zip(image_paths, processed_images):
            output_path = output_folder / path.name
            cv2.imwrite(str(output_path), image)

In [ ]:
preprocess_pipeline(parent_dir, child_dir, face_detection_model_path, padding=face_detection_padding, size=image_resize_dim, visualize=False, overwrite_existing=overwrite_existing)

### User-defined PyTorch Dataset Class

In [22]:
class KinshipVerificationPairs(Dataset):
  def __init__(self, parent_dir, child_dir, transform=None):
    self.parents = sorted(parent_dir.glob("*")) # Sort the file paths to ensure consistent ordering
    self.children = sorted(child_dir.glob("*")) # Return a list of all file paths in the directory
    self.transform = transform

    self.popUnused()

    self.positive_pairs = [(i, i) for i in range(len(self.parents))]
    self.negative_pairs = []
    self.make_negative_pairs()

    pos_df = pd.DataFrame(self.positive_pairs, columns=["parent_idx", "child_idx"])
    pos_df["label"] = 1

    neg_df = pd.DataFrame(self.negative_pairs, columns=["parent_idx", "child_idx"])
    neg_df["label"] = 0

    self.train_df = pd.concat([pos_df, neg_df], ignore_index=True)


  def popUnused(self):
    self.parents.pop()
    self.children.pop()
    print(self.__len_parents__())
    print(self.__len_children__())


  def __len_parents__(self):
    return len(self.parents)

  def __len_children__(self):
    return len(self.children)

  def __len__(self):
    return len(self.train_df)


  def __getitem__(self, idx):
    row = self.train_df.iloc[idx]
    p_idx = row["parent_idx"]
    c_idx = row["child_idx"]
    label = row["label"]

    parent_img = Image.open(self.parents[p_idx])
    child_img = Image.open(self.children[c_idx])

    if self.transform:
        parent_img = self.transform(parent_img)
        child_img = self.transform(child_img)

    return parent_img, child_img, label

  def openImage(self, fromChildren, idx):
    if(fromChildren):
      img = Image.open(self.children[idx])
    else:
      img = Image.open(self.parents[idx])
    display(img)

  def openImagePair(self, pair_list, idx):
    parent_img = Image.open(self.parents[pair_list[idx][0]])
    child_img = Image.open(self.children[pair_list[idx][1]])
    plt.subplot(1,2,1)
    plt.imshow(parent_img)
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(child_img)
    plt.axis("off")

    plt.show()



  def make_negative_pairs(self):
    used = set()
    for p in range(len(self.parents)):
      c = random.randrange(len(self.children))
      while c == p or (p, c) in used:
          c = random.randrange(len(self.children))
      used.add((p, c))
      self.negative_pairs.append((p, c))

def detect_face(self, image_path, visualize=False):
  base_options = python.BaseOptions(model_asset_path='blaze_face_short_range.tflite')
  options = vision.FaceDetectorOptions(base_options=base_options,
                                        running_mode=vision.RunningMode.IMAGE,
                                        min_detection_confidence=0.5,
                                        num_faces=1)
  detector = vision.FaceDetector.create_from_options(options)

  image = cv2.imread(image_path)
  image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  
  mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
  detection_result = detector.detect(mp_image)

  if visualize:
    image_copy = np.copy(mp_image.numpy_view())
    annotated_image = visualize(image_copy, detection_result)

    plt.imshow(annotated_image)
    plt.axis("off")
    plt.show()
    
  return detection_result


### Testing Dataset Class

In [31]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
berkeley_dataset = KinshipVerificationPairs(parent_dir, child_dir, transform=transform)
#berkeley_dataset.openImage(1, 1)

143
143


In [51]:
# berkeley_dataset.openImagePair(berkeley_dataset.positive_pairs, 10)


In [52]:
# berkeley_dataset.openImagePair(berkeley_dataset.negative_pairs, 2)

# Self-notes

In [ ]:
torch.reshape()

#### Define the face detection model using MediaPipe's FaceDetector. 

In [53]:
# # This model will be used to detect faces in the input images and extract facial features for the kinship verification task.
# base_options = python.BaseOptions(model_asset_path='blaze_face_short_range.tflite') #Model asset path is the path to the .tflite (TensorFlow Lite) file of the model you want to use. This file is the face detection model. Chose the short range model since the images in the dataset are close up shots of faces.
# options = vision.FaceDetectorOptions(
#     base_options=base_options, # Passing the base options to the FaceDetectorOptions. This is necessary to specify the model asset path and any other base options that may be needed.
#     running_mode=vision.RunningMode.IMAGE, # Setting the running mode to IMAGE since we are working with still images.
#     min_detection_confidence=0.5, # Minimum confidence score for face detection to be considered successful. This means that the model will only consider detections with a confidence score of 0.5 or higher as valid detections.
# )
# detector = vision.FaceDetector.create_from_options(options) # Creating the FaceDetector object using the options we just defined. This object will be used to perform face detection on the input images.


# image = cv2.imread(berkeley_dataset.parents[0]) # cv2.imread is used to read the image, which is in BGR format by default. Returns a NumPy array representing the image.
# img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Converting the image from BGR to RGB format using cv2.cvtColor because the Mediapipe model expects images in RGB format.


# mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb) # Creating a Mediapipe Image object from the RGB image. The image_format parameter specifies the format of the input image, which is SRGB in this case. The data parameter is the actual image data in the form of a NumPy array.


# detection_result = detector.detect(mp_image) # Performs face detection, and returns the detection results, which include the bounding boxes and keypoints of the detected faces in the image.

# image_copy = np.copy(mp_image.numpy_view()) # Creating a copy of the original image data as a NumPy array. This is done to avoid modifying the original image data when we draw the bounding boxes and keypoints on it for visualization purposes.
# annotated_image = visualize(image_copy, detection_result) # Calling the visualize function to draw the bounding boxes and keypoints on the copied image using the detection results.

# plt.imshow(annotated_image)
# plt.axis("off")
# plt.show()